<a href="https://colab.research.google.com/github/aristidekanamugire/Titanic-data-cleaning/blob/main/Lab_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Titanic Data Munging and Cleaning**

Task 1: Loading & Inspecting

In [ ]:
# Task 1: Load & Inspect Titanic Dataset

import pandas as pd

# Load from Google Drive path
data_path = "/content/drive/MyDrive/titanic.csv"
df = pd.read_csv(data_path)

# Display first 10 rows
print("First 10 rows:")
display(df.head(10).style.set_caption("**First 10 Rows of Titanic Dataset**").set_table_styles(
    [{'selector': 'caption', 'props': [('font-weight', 'bold'), ('font-size', '14px')]}]
))

# Info summary
print("\nDataFrame Info:")
print(df.info())

# Statistical summary
print("\nStatistical Summary:")
display(df.describe().style.set_caption("**Statistical Summary**").set_table_styles(
    [{'selector': 'caption', 'props': [('font-weight', 'bold'), ('font-size', '14px')]}]
))

# Missing values count
print("\nMissing values in each column:")
missing_table = df.isnull().sum().to_frame("Missing Count")
display(missing_table.style.set_caption("**Missing Values Per Column**").set_table_styles(
    [{'selector': 'caption', 'props': [('font-weight', 'bold'), ('font-size', '14px')]}]
))



Task 2 — Missing Data Imputation

In [ ]:
# Task 2: Handle Missing Data

import numpy as np

# % missing for Age
age_missing_pct = df['Age'].isnull().mean() * 100
print(f"Missing Age Percentage: {age_missing_pct:.2f}%")

# Impute Age by median grouped by Pclass and Sex (only if missing exists)
if df['Age'].isnull().sum() > 0:
    df['Age'] = df.groupby(['Pclass', 'Sex'])['Age'].transform(
        lambda x: x.fillna(x.median())
    )

# Fill Embarked with mode if missing exists
if df['Embarked'].isnull().sum() > 0:
    embarked_mode = df['Embarked'].mode()[0]
    df['Embarked'] = df['Embarked'].fillna(embarked_mode)

# Drop Cabin if exists
df = df.drop(columns=['Cabin'], errors='ignore')

print("\nMissing values after imputation:")
missing_table = df.isnull().sum().to_frame("Missing Count")
display(missing_table.style.set_caption("**Missing Values After Imputation**").set_table_styles(
    [{'selector': 'caption', 'props': [('font-weight', 'bold'), ('font-size', '14px')]}]
))


Task 3 — Outlier Detection & Treatment

In [ ]:
# Task 3: Outlier Detection for Age and Fare

import matplotlib.pyplot as plt

# Boxplots
df[['Age', 'Fare']].plot(kind='box', subplots=True, layout=(1,2), figsize=(12,5))
plt.show()

# Compute IQR bounds
def iqr_bounds(series):
    Q1, Q3 = series.quantile([0.25, 0.75])
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return lower, upper

outlier_counts = {}
for col in ['Age', 'Fare']:
    low, high = iqr_bounds(df[col])
    outliers = df[(df[col] < low) | (df[col] > high)]
    outlier_counts[col] = len(outliers)

    # Treat outliers by capping
    lower_cap = df[col].quantile(0.01)
    upper_cap = df[col].quantile(0.99)
    df[col] = np.clip(df[col], lower_cap, upper_cap)

# Show outlier counts table
outlier_table = pd.DataFrame.from_dict(outlier_counts, orient="index", columns=["Outlier Count"])
display(outlier_table.style.set_caption("**Detected Outliers Before Treatment**").set_table_styles(
    [{'selector': 'caption', 'props': [('font-weight', 'bold'), ('font-size', '14px')]}]
))


Task 4 — Data Compatibility & Unification

In [ ]:
# Task 4: Feature Unification & Encoding

# Extract Titles
df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)

# Group rare titles
rare_titles = df['Title'].value_counts()[df['Title'].value_counts() < 10].index
df['Title'] = df['Title'].replace(rare_titles, 'Other')

# Convert categorical columns
df['Sex'] = df['Sex'].astype('category')
df['Embarked'] = df['Embarked'].astype('category')

# Rename columns
df.columns = df.columns.str.lower().str.replace(' ', '_')

# One-hot encode categorical
df = pd.get_dummies(df, columns=['sex', 'embarked', 'title'], drop_first=True)

display(df.head(10).style.set_caption("**DataFrame After Encoding**").set_table_styles(
    [{'selector': 'caption', 'props': [('font-weight', 'bold'), ('font-size', '14px')]}]
))


Task 5 — Normalization and Z-Scores

In [ ]:
# Task 5: Normalization with Z-Scores

from scipy.stats import zscore
import numpy as np

def safe_zscore(series):
    if series.std() == 0 or series.isnull().all():
        return pd.Series([0]*len(series), index=series.index)  # fallback to 0
    return zscore(series)

# Compute z-scores safely
if 'age' in df.columns:
    df['age_z'] = safe_zscore(df['age'])
if 'fare' in df.columns:
    df['fare_z'] = safe_zscore(df['fare'])

# Compare histograms only if data exists
fig, axes = plt.subplots(1, 2, figsize=(12,5))

# Raw distributions
if 'age' in df.columns and not df['age'].isnull().all():
    axes[0].hist(df['age'].dropna(), bins=20, alpha=0.7, label="Age")
if 'fare' in df.columns and not df['fare'].isnull().all():
    axes[0].hist(df['fare'].dropna(), bins=20, alpha=0.7, label="Fare")
axes[0].set_title("Raw Distributions")
axes[0].legend()

# Z-scored distributions
if 'age_z' in df.columns and not df['age_z'].isnull().all():
    axes[1].hist(df['age_z'].dropna(), bins=20, alpha=0.7, label="Age (Z)")
if 'fare_z' in df.columns and not df['fare_z'].isnull().all():
    axes[1].hist(df['fare_z'].dropna(), bins=20, alpha=0.7, label="Fare (Z)")
axes[1].set_title("Z-Scored Distributions")
axes[1].legend()

plt.show()

# Show a bold sample comparison table
z_table = df[['age', 'fare', 'age_z', 'fare_z']].head(10)
display(z_table.style.set_caption("**Sample of Raw vs. Z-Scored Values (Fixed)**").set_table_styles(
    [{'selector': 'caption', 'props': [('font-weight', 'bold'), ('font-size', '14px')]}]
))


Task 6 — Reproducible Cleaning Pipeline

In [ ]:
# Task 6: Cleaning Pipeline

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline

class TitanicCleaner(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = X.copy()

        # Age imputation
        if df['Age'].isnull().sum() > 0:
            df['Age'] = df.groupby(['Pclass', 'Sex'])['Age'].transform(
                lambda x: x.fillna(x.median())
            )

        # Embarked imputation
        if df['Embarked'].isnull().sum() > 0:
            df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

        # Drop Cabin safely
        df = df.drop(columns=['Cabin'], errors='ignore')

        # Titles
        df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
        rare_titles = df['Title'].value_counts()[df['Title'].value_counts() < 10].index
        df['Title'] = df['Title'].replace(rare_titles, 'Other')

        # Rename columns
        df.columns = df.columns.str.lower().str.replace(' ', '_')

        # One-hot encode
        df = pd.get_dummies(df, columns=['sex', 'embarked', 'title'], drop_first=True)

        # Cap outliers
        for col in ['age', 'fare']:
            if col in df.columns:
                lower_cap = df[col].quantile(0.01)
                upper_cap = df[col].quantile(0.99)
                df[col] = np.clip(df[col], lower_cap, upper_cap)

        # Add z-scores
        if 'age' in df.columns: df['age_z'] = zscore(df['age'])
        if 'fare' in df.columns: df['fare_z'] = zscore(df['fare'])

        return df

# Run pipeline
pipeline = Pipeline([
    ('cleaner', TitanicCleaner())
])

cleaned_df = pipeline.fit_transform(pd.read_csv(data_path))

# Export cleaned dataset
cleaned_df.to_csv("/content/drive/MyDrive/cleaned_titanic.csv", index=False)
print(" Cleaned data saved as cleaned_titanic.csv")

display(cleaned_df.head(10).style.set_caption("**Final Cleaned Titanic Data**").set_table_styles(
    [{'selector': 'caption', 'props': [('font-weight', 'bold'), ('font-size', '14px')]}]
))
